In [1]:
import sys
sys.path.insert(0, '../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

## Load EEG and preprocesses in same fashion as Alice dataset

Loads EEG data from data_preprocessed but adds preprocessing steps to align with preprocessing of Alice dataset.

In [2]:
SUBJECTS = helper_functions.fuglsang_get_subjects()

In [3]:
def preprocess_eeg(info, subject, padded=False):
    """
    Preprocess and save EEG data for a single subject.
    Saves both concatenated EEG and individual trial EEG files.

    Args:
        subject: Subject ID.
        padded:  Whether to also save a padded version.
    """
    dst_dir = FUGLSANG_EEG_DIR / subject
    dst_dir.mkdir(exist_ok=True, parents=True)

    suffix       = "_padded" if padded else ""
    eeg_path     = dst_dir / f"{subject}_eeg{suffix}.pickle"
    trials_dir   = dst_dir / f"trials{suffix}"

    # check if everything already exists
    if eeg_path.exists() and trials_dir.exists():
        print(f"{subject}: EEG exists, skipping.")
        return

    print(f"{subject}: preprocessing EEG...")

    mat  = scipy.io.loadmat(
        FUGLSANG_DATA_PREPROC / f"{subject}_data_preproc.mat",
        squeeze_me=True, struct_as_record=False
    )
    data = mat["data"]

    trials = []

    for trial_idx, trial in enumerate(data.eeg):
        eeg = np.array(trial).T
        raw = mne.io.RawArray(eeg * 1e-6, info, verbose=False)

        raw.set_eeg_reference(['EXG1', 'EXG2'])
        raw.filter(BANDPASS_FILTER_LOW, BANDPASS_FILTER_HIGH, verbose=False)

        eeg_ndvar = eelbrain.load.mne.raw_ndvar(raw)

        if padded:
            eeg_ndvar = eelbrain.pad(
                eeg_ndvar,
                tstart=-PADDING_ONSET,
                tstop=eeg_ndvar.time.tstop + PADDING_OFFSET
            )

        trials.append(eeg_ndvar)

    # save concatenated (existing behaviour, used by subject-level AAD)
    eeg_concat = eelbrain.concatenate(trials, name=f"{subject}_eeg{suffix}")
    eelbrain.save.pickle(eeg_concat, eeg_path)
    print(f"  ✓ Saved concatenated: {eeg_path}")

    # save individual trials (needed for trial-level AAD)
    trials_dir.mkdir(exist_ok=True, parents=True)
    for trial_idx, eeg_ndvar in enumerate(trials):
        trial_path = trials_dir / f"trial_{trial_idx:02d}{suffix}.pickle"
        eelbrain.save.pickle(eeg_ndvar, trial_path)

    print(f"  ✓ Saved {len(trials)} trial files to {trials_dir}")

In [4]:
# MNE setup
montage   = mne.channels.make_standard_montage('biosemi64')
ch_names  = montage.ch_names + ['EXG1', 'EXG2']
ch_types  = ['eeg'] * 64 + ['misc', 'misc']
info      = mne.create_info(ch_names, EEG_SAMPLING_RATE, ch_types)
info.set_montage(montage, on_missing='ignore')

for subject in SUBJECTS:
    preprocess_eeg(info, subject, padded=False)
    preprocess_eeg(info, subject, padded=True)

print("Done preprocessing EEG.")

S1: preprocessing EEG...
  ✓ Saved concatenated: /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S1/S1_eeg.pickle
  ✓ Saved 60 trial files to /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S1/trials
S1: preprocessing EEG...
  ✓ Saved concatenated: /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S1/S1_eeg_padded.pickle
  ✓ Saved 60 trial files to /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S1/trials_padded
S2: preprocessing EEG...
  ✓ Saved concatenated: /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S2/S2_eeg.pickle
  ✓ Saved 60 trial files to /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S2/trials
S2: preprocessing EEG...
  ✓ Saved concatenated: /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S2/S2_eeg_padded.pickle
  ✓ Saved 60 trial files to /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/eeg/S2/trials_padded
S3: preprocessing EEG...
  ✓ Saved concatenated: /Us

In [5]:
# SANITY CHECK: Check dimensions of preprocessed EEG
subject = SUBJECTS[0]
for padded in [False, True]:
    suffix   = "_padded" if padded else ""
    eeg_path = FUGLSANG_EEG_DIR / subject / f"{subject}_eeg{suffix}.pickle"
    if eeg_path.exists():
        print(f"  ✓ eeg{suffix}: {eelbrain.load.unpickle(eeg_path)}")
    else:
        print(f"  ✗ eeg{suffix}: MISSING")

  ✓ eeg: <NDVar 'S1_eeg': 64 sensor, 192000 time>
  ✓ eeg_padded: <NDVar 'S1_eeg_padded': 64 sensor, 196260 time>
